# 06 - Feeding Detection
Detect feeding events using pose-based heuristics.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/LightningPoseTrack.git"
GIT_BRANCH = "main"

DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"
DRIVE_BEHAVIOR_OUTPUTS = f"{DRIVE_ROOT}/behavior_outputs"

# Feeder position (pixels) — set these!
FEEDER_X = 300  # TODO: update
FEEDER_Y = 400  # TODO: update
FEEDER_DIST_THRESHOLD = 50.0   # pixels
HEADING_ALIGNMENT_TOLERANCE = 45.0  # degrees
MIN_FEEDING_DURATION = 2.0  # seconds
FPS = 30.0
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet pandas numpy pyarrow

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np

# Reuse the feeding detection logic from src
from src.behaviors.feeding import detect_feeding_events, compute_feeding_summary
from src.features.orientation import compute_heading
from src.pose.clean_pose import clean_pose_df

pose_dir = Path(DRIVE_POSE_OUTPUTS)
behavior_dir = Path(DRIVE_BEHAVIOR_OUTPUTS)
behavior_dir.mkdir(parents=True, exist_ok=True)

pose_files = list(pose_dir.rglob("*.parquet"))
print(f"Processing {len(pose_files)} pose files for feeding detection...")

all_events = []
for pose_file in tqdm(pose_files, desc="Detecting feeding"):
    df = pd.read_parquet(pose_file)
    df_clean = clean_pose_df(df)
    df_clean["heading_rad"] = compute_heading(df_clean)

    angle_tol_rad = np.radians(HEADING_ALIGNMENT_TOLERANCE)
    events = detect_feeding_events(
        df_clean,
        feeder_x=FEEDER_X,
        feeder_y=FEEDER_Y,
        fps=FPS,
        distance_threshold=FEEDER_DIST_THRESHOLD,
        angle_tolerance=angle_tol_rad,
        min_duration_sec=MIN_FEEDING_DURATION,
    )

    # Add metadata
    rel = pose_file.relative_to(pose_dir)
    parts = str(rel).split("/")
    session = parts[0] if len(parts) > 1 else "unknown"
    camera = 0
    import re
    cam_match = re.search(r"_cam(\d+)_", pose_file.name)
    if cam_match:
        camera = int(cam_match.group(1))
    if not events.empty:
        events["session"] = session
        events["camera"] = camera
        events["video"] = pose_file.name
        all_events.append(events)

    # Save per-video events
    out_path = behavior_dir / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path = out_path.with_name(out_path.stem.replace("_pose", "_feeding_events") + ".parquet")
    events.to_parquet(out_path)

if all_events:
    master = pd.concat(all_events, ignore_index=True)
    master_path = behavior_dir / "feeding_events.parquet"
    master.to_parquet(master_path)
    print(f"\nTotal feeding events detected: {len(master)}")
    print(master.head(10))
else:
    print("No feeding events detected. Adjust thresholds.")

In [ ]:
if all_events:
    master = pd.read_parquet(behavior_dir / "feeding_events.parquet")
    print("=== Feeding Summary ===")
    print(f"Total events: {len(master)}")
    print(f"Mean duration: {master['duration_sec'].mean():.1f}s")
    print(f"Total feeding time: {master['duration_sec'].sum():.1f}s")
    print(f"\nPer session:")
    print(master.groupby('session').agg(
        events=('duration_sec', 'count'),
        total_time_s=('duration_sec', 'sum')
    ))